# 등굣길 (프로그래머스, Level 3)

## 문제 요약
- `m x n` 격자에서 집(1,1)에서 학교(m,n)까지 오른쪽/아래로만 이동
- `puddles`에 있는 칸은 물 웅덩이라 지나갈 수 없음
- **최단 경로의 개수**를 `1_000_000_007`로 나눈 나머지를 반환

## 핵심 아이디어 — DP (동적 계획법)
어떤 칸 `(x, y)`에 도달하는 최단 경로는 두 가지 방법뿐이다:
1. 왼쪽 칸 `(x-1, y)`에서 오른쪽으로 이동
2. 위쪽 칸 `(x, y-1)`에서 아래로 이동

따라서 점화식은:
```
dp[y][x] = dp[y][x-1] + dp[y-1][x]   (웅덩이면 dp[y][x] = 0)
```
시작점 `dp[1][1] = 1`, 웅덩이는 갈 수 없으니 0으로 고정하고 테이블을 채우면 끝.

## 주의할 함정들 ⚠️
1. **좌표 뒤집기**: 문제/`puddles`는 `(열, 행) = (x, y)` 순서. 배열 인덱스는 `[행][열] = [y][x]`라서 `puddle[0] = x, puddle[1] = y`임을 헷갈리지 말 것.
2. **오버플로**: 경로 수가 매우 커지므로 매 연산마다 `1_000_000_007`로 나머지를 저장.
3. **웅덩이 좌표 1-based**: 그대로 인덱스로 쓰려면 패딩을 붙이는 게 편함 (`(n+1) x (m+1)` 테이블, 0행/0열은 경로 0 → 경계 조건 공짜 처리).

## 자바 풀이 (먼저 눈으로 이해 → 같은 로직을 파이썬으로 직접 구현!)

```java
class Solution {
    public int solution(int m, int n, int[][] puddles) {
        // dp[행 y][열 x], 패딩으로 인덱스 1부터 사용
        int[][] dp = new int[n + 1][m + 1];
        boolean[][] puddle = new boolean[n + 1][m + 1];

        // puddles는 (x, y) 형태 → [y][x]에 체크
        for (int[] p : puddles) {
            puddle[p[1]][p[0]] = true;
        }

        dp[1][1] = 1;
        int MOD = 1_000_000_007;

        for (int y = 1; y <= n; y++) {
            for (int x = 1; x <= m; x++) {
                if (puddle[y][x]) {   // 웅덩이는 경로 0
                    dp[y][x] = 0;
                    continue;
                }
                if (x == 1 && y == 1) continue; // 시작점은 이미 1
                dp[y][x] = (dp[y][x - 1] + dp[y - 1][x]) % MOD;
            }
        }
        return dp[n][m];
    }
}
```

## 흐름 정리 (파이썬으로 옮길 때 이대로 따라가면 됨)
1. `(n+1) x (m+1)` DP 테이블 준비 (1-based 인덱스)
2. 웅덩이 좌표 표시 — 이때 `(x, y)`를 `[y][x]`로 반전시켜 인덱스에 넣기
3. 시작점에 1 넣고, 위→아래 / 왼→오 순서로 `왼쪽 + 위쪽` 채우기
4. 채울 때마다 MOD로 나머지 → `dp[n][m]` 반환, 시간복잡도 O(n·m)

In [5]:
# 내가 자바 코드 보고 따라해본 구현
def solution(m, n, puddles):
    dp = [[0] * (m + 1) for _ in range(n+1)]
        
    puddles = {(py, px) for px, py in puddles}
    mod = 1000000007
    dp[1][1] = 1
    for y in range(1,n+1,1):
        for x in range(1,n+1,1):
            if (x, y) in puddles:
                
                continue
            elif x == 1 and y == 1:
                continue
            else:
                dp[y][x] = (dp[y][x -1] + dp[y-1][x]) % mod
    return dp[n][m]
# assert solution(4, 3, [[2, 2]]) == 4    # 예제: 웅덩이 (2,2), 답 4
assert solution(1, 1, []) == 1          # 집 = 학교
# assert solution(2, 2, [[1, 2], [2, 1]]) == 0  # 모두 막혀서 못 감

## 피드백 (자바 코드 대조해보기) 🔍

전체 구조(테이블 채우기 접근)는 자바 로직을 잘 옮겼는데, 번역 과정에서 버그 몇 개:

| # | 코드 | 문제 | 고치기 |
|---|------|------|--------|
| 1 | `dp = []` | 빈 리스트 — `dp[y][x]` 인덱싱하면 바로 IndexError. 자바의 `new int[n+1][m+1]` (0으로 채워진 테이블)에 대응되는 게 없음 | `dp = [[0] * (m + 1) for _ in range(n + 1)]` |
| 2 | `ex, ey = puddles` | `puddles`는 좌표들의 **리스트** (웅덩이 여러 개 가능). 리스트 전체 언패킹은 TypeError. 자바의 `for (int[] p : puddles)` 순회가 빠짐 | `puddle = {(py, px) for px, py in puddles}` — 순회 + (x,y)→(y,x) 반전 한 번에! |
| 3 | `range(1, n, 1)` 두 번 | (a) `range` 끝값은 미포함 → 마지막 행/열을 안 돌고 학교(m,n)에 도달 못함 (b) x는 **m**까지인데 둘 다 n | `range(1, n + 1)` / `range(1, m + 1)` |
| 4 | `dp[y][x] = 0:` | 값 뒤의 `:` — 문법 오류. `:`는 `if`/`def` 같은 **문의 끝**에만 | `continue`만 하면 됨 (테이블이 0으로 시작하니 대입 자체가 불필요) |
| 5 | `if ex == x and ey == y` | 자바는 웅덩이를 미리 `boolean[][]`에 표시 후 O(1) 조회. 여러 웅덩이와 매번 비교는 느리고 번거로움 | `if (x, y) in puddle:` — #2의 집합 활용 |

잘한 점 ✅: 시작점 continue 처리, 점화식 `dp[y][x-1] + dp[y-1][x]`, MOD 적용까지 자바 로직 그대로.

### 수정된 코드 (직접 고쳐본 뒤에 비교!)

```python
def solution(m, n, puddles):
    MOD = 1_000_000_007
    dp = [[0] * (m + 1) for _ in range(n + 1)]      # ① 테이블
    puddle = {(py, px) for px, py in puddles}        # ② 반전 저장

    dp[1][1] = 1                                     # ③ 시작점
    for y in range(1, n + 1):
        for x in range(1, m + 1):
            if (x, y) in puddle:
                continue                             # 웅덩이는 0 유지
            if x == 1 and y == 1:
                continue
            dp[y][x] = (dp[y][x - 1] + dp[y - 1][x]) % MOD
    return dp[n][m]
```

### 연습용 테스트
```python
assert solution(4, 3, [[2, 2]]) == 4    # 예제: 웅덩이 (2,2), 답 4
assert solution(1, 1, []) == 1          # 집 = 학교
assert solution(2, 2, [[1, 2], [2, 1]]) == 0  # 모두 막혀서 못 감
```

## C 풀이 (같은 로직, C 문법으로)

```c
#include <stdio.h>
#include <stdlib.h>

#define MOD 1000000007

int solution(int m, int n, int** puddles, int puddlesSize) {
    // ① (n+1) x (m+1) 테이블 동적 할당 (0으로 초기화)
    long long** dp = (long long**)calloc(n + 1, sizeof(long long*));
    char** puddle = (char**)calloc(n + 1, sizeof(char*));
    for (int i = 0; i <= n; i++) {
        dp[i] = (long long*)calloc(m + 1, sizeof(long long));
        puddle[i] = (char*)calloc(m + 1, sizeof(char));
    }

    // ② 웅덩이 표시: puddles[i] = {x, y} → puddle[y][x] = 1
    for (int i = 0; i < puddlesSize; i++) {
        puddle[puddles[i][1]][puddles[i][0]] = 1;
    }

    // ③ 시작점
    dp[1][1] = 1;

    // ④ 테이블 채우기: 위→아래, 왼→오
    for (int y = 1; y <= n; y++) {
        for (int x = 1; x <= m; x++) {
            if (puddle[y][x]) continue;          // 웅덩이는 0 유지
            if (x == 1 && y == 1) continue;      // 시작점은 이미 1
            /* int끼리 더하면 오버플로 위험 →(long long)으로 승격 후 더하고 MOD */
            dp[y][x] = (dp[y][x - 1] + dp[y - 1][x]) % MOD;
        }
    }

    int answer = (int)dp[n][m];
    // ⑤ 메모리 해제 (C는 가비지 컬렉터 없음 — 반드시 free)
    for (int i = 0; i <= n; i++) {
        free(dp[i]);
        free(puddle[i]);
    }
    free(dp);
    free(puddle);
    return answer;
}
```

## 코드 상세 해설 (자바 기준, 모든 언어 같은 개념)

### ① 테이블 생성
```java
int[][] dp = new int[n + 1][m + 1];
boolean[][] puddle = new boolean[n + 1][m + 1];
```
- `dp[y][x]` = "(x, y) 칸에 도달하는 최단 경로의 수". 문제를 그대로 셀 단위 미니 문제로 쪼갠 것.
- 왜 `+1`이나? → 좌표가 1부터 시작(1-based). 인덱스 0은 **버퍼(패딩)**. 덕분에:
  - `dp[y][x-1]`에서 `x=1`일 때 `dp[?][0]`을 참조하는데, 0열은 0이므로 **if문으로 경계 검사 안 해도 됨**
  - 자바/C는 배열이 자동으로 0/false 초기화 → 0행, 0열, 웅덩이 칸을 따로 초기화할 필요 없음
- `puddle`은 DP 테이블과 따로 둔 **표시용 지도**: "이 칸 지나갈 수 없음" 정보만 담당. DP와 상태를 분리하면 실수가 줄어듦.

### ② 웅덩이 등록 — 좌표 반전이 핵심
```java
for (int[] p : puddles) {
    puddle[p[1]][p[0]] = true;
}
```
- 문제에서 웅덩이 좌표는 `(x, y)` = `(열, 행)` 순서로 준다.
- 그런데 배열 인덱스는 `[행][열]` 순서. 그래서 `p[0]`(x=열)은 **뒤**, `p[1]`(y=행)은 **앞**에 들어가야 함 → `puddle[p[1]][p[0]]`.
- 이걸 놓치면 웅덩이가 "대각선 뒤집힌 위치"에 세팅되어 예제조차 틀리게 나옴. 등굣길 제일 흔한 실수.

### ③ 시작점 시딩 (seed)
```java
dp[1][1] = 1;
```
- 집에서 집으로 가는 경로는 "아무것도 안 움직이는 경로" 하나뿐
- 점화식은 판다(`dp[1][1] = dp[0][1] + dp[1][0] = 0 + 0 = 0`)가 아니라 **깔아주는 것이 필요** → 이러한 기저값(base case) 없이는 전체가 0으로 계산돼버림.
- 그리고 아래 루프에서 `x==1 && y==1`은 다시 계산하지 않도록 함) → 시딩 값 1이 지워지지 않도록.

### ④ 점화식 루프 — 왜 이 순서일까?
```java
dp[y][x] = (dp[y][x - 1] + dp[y - 1][x]) % MOD;
```
- 오른쪽/아래로만 이동하므로 "(x,y) 도달" = 왼쪽 도달 + 위쪽 도달. 두 경우가 겹치지 않음(마지막 이동 방향으로 구분됨) → 단순 합으로 정답.
- 위→아래, 왼→오 순서로 돌면 채우려는 칸의 왼쪽·위쪽 값이 **항상 먼저 완성되어 있음**
  (DP의 2차원 표준 순회). 반대로 돌면 쓰레기값 0을 참조하게 됨.
- 웅덩이 칸은 `continue`로 점프 → `dp`값이 0으로 유지되고, "웅덩이를 경유하는 경로는 자동으로 합산에서 제외"됨. 웅덩이 주변 칸들부터 시작점처럼 0을 전해준다는 느낌.
- `% MOD`: 두 값이 각각 MOD 미만이고(long 자릿수 생각) `int`는 overflow 위험성이 있으므로, 자바는 `int+int`가 곧바로 오버플로할 수 있어 매번 나머지를 취함. C에서는 `long long`에 담아 더한 후 나머지를 취하며 견고하게 처리 (위 코드 참고). 마지막에만 나누면 누적 과정에서 이미 오버플로됨 → **매 칸마다 MOD**.

### ⑤ 반환
```java
return dp[n][m];
```
- 문제 좌표계에서 학교가 `(m, n)` = (열 m, 행 n)이고 배열 인덱스로는 `dp[행][열] = dp[n][m]`. 여기서도 x/y 반전이 그대로 적용됨!

### 전체 흐름 요약
```
표 만들기 → 웅덩이 찍기 → 시작점 시딩 → 왼쪽+위쪽 합으로 채우기(매번 MOD) → 우하단 값이 정답
```
- 시간 O(n·m), 공간 O(n·m). 같은 행만 보면 되므로 1차원 배열로 O(min(m,n)) 공간까지 줄일 수도 있음 (심화).

### 언어 간 차이 한눈에
| 항목 | 자바 | C | 파이썬 |
|------|------|---|--------|
| 배열 | `new int[n+1][m+1]` (자동 0) | `calloc` 후 `free` 필수 | `[[0]*(m+1) for _ in range(n+1)]` |
| 오버플로 | `int` + `int` 위험 → 매번 `%` | `long long` 승격 후 `%` | 자동 큰수, 그래도 매번 `%` (성능) |
| 웅덩이 | `boolean[][]` | `char[][]` (0/1) | `set{(py, px)}` → `in` |

In [6]:
def solution(m, n, puddles):
    if [m, n] in puddles:
        return 0
    if m == 1 and n == 1:
        return 1
    if m == 1: # 위쪽 행이 1 일때는 왼쪽으로만 이동 가능
        return solution(m, n-1, puddles)
    if n == 1: # 왼쪽 열이 1 일때는 위 쪽으로만 이동 가능
        return solution(m-1, n, puddles)

    # 그 외 경우 (위 쪽에서 오는 경로 + 왼쪽에서 오는 경로)
    return solution(m, n-1, puddles) + solution(m-1, n, puddles)

m, n, puddles = 4, 3, [[2,2]]
print(puddles[0][0])
print(solution(m,n, puddles))

2
4
